# Does host vocabulary *start* uptake or *grow* it?  K1 margins, K3 field boundary, size-correct inference

**Label: POST-CONFIRMATION EXPLORATORY.** This notebook is a runnable demo of the evaluation artifact `eval.py`
(iteration 5, `gen_art_evaluation_6`). The research asks how emerging scientific concepts enter and spread through new
host subfields of an OpenAlex-based knowledge network. An earlier confirmed result (**D2**) is that a concept whose
*entry vocabulary leans on the host subfield's own terms* (`A_cont`) sees more **newcomer uptake** in that subfield
(`Y_strict`, a count over the 5 years after entry). The evaluation asks three follow-up questions:

| Question | What is estimated |
|---|---|
| **K1: which margin?** | Does `A_cont` raise the probability that *any* uptake starts (extensive margin: LPM / FE-logit / log-link on `1[Y>=1]`), or only *how much* follows (intensive margin: PPML on `Y>=1`)? Plus a log-link decomposition with a concept-cluster bootstrap, and a threshold ladder (`Y>=1`, `Y>=3`, `Y>=5`, `EST_bin`). |
| **K3: field boundary?** | Does the effect stop at Physics/Astro-origin concepts? (pooled screen + held-out Wald test) |
| **INFERENCE** | Are the headline p values size-correct with few clusters? (randomization-t, Freedman-Lane, wild cluster score bootstrap with Webb / Rademacher weights) |

**What this demo runs.** `eval.py` itself is only an orchestrator that runs six stage scripts as subprocesses
(freeze → smoke → run → audit → report → placebo). The demo copies the estimation code of the **run** stage
(`k/k_lib.py`, `k/k_run.py` and the vendored `d2/src/ppml.py`, `d2/src/models.py`) into cells, and runs **K1** and
**INFERENCE** on the frozen **held-out fold** (93 concepts, 1,097 entry events). That fold is the one whose significance was in
question. K3 needs the screen fold as well, so K3 is shown from the artifact's stored results, as are the
screen / MeSH folds and the 3-fold IVW pooling.

The functions are the original code. The only changes are: data comes from `mini_demo_data.json` instead of parquet
files, the process pool is replaced by a plain loop (spawned workers cannot import notebook-defined functions), and
the draw counts come from the config cell.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# numpy, pandas, scipy, matplotlib — pre-installed on Colab, install locally only (Colab's exact versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# ---- original import blocks (eval.py, k/k_lib.py, k/k_run.py), unchanged apart from dropping the process-pool imports
from __future__ import annotations

import os
for _v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):  # one BLAS thread per worker process
    os.environ[_v] = "1"
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from loguru import logger
from scipy import sparse, stats
from scipy.sparse.linalg import splu

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# ---- additional imports for the notebook
import types
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-8892b7-occupancy-is-not-integration-for-new/main/round-5/evaluation-6/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
meta = data["metadata"]
examples = data["datasets"][0]["examples"]
print(meta["title"])
print(f"dataset: {data['datasets'][0]['dataset']} | {len(examples)} concepts (examples), "
      f"{sum(e['n_entries'] for e in examples)} entry events")
print("frozen spec sha256:", meta["spec"]["sha256"])
pd.DataFrame([{k: v for k, v in e.items() if k != "entries"} for e in examples]).head()

## Configuration

The draw counts are the only tunable parameters. The original values come from the frozen spec (`k13_spec.json`) and
are noted beside each one. The defaults below **are** the original values: the whole notebook runs in about 4 minutes
on 2 CPUs (about 70 s bootstrap + 120 s randomization). To make a quicker pass, lower them (for example 100 / 200 / 999).
A single PPML fit on the held-out fold takes about 0.03 s. The randomization and bootstrap
draws therefore dominate the runtime: one bootstrap draw is 3 PPML fits, and one randomization draw is 2 PPML + 2 LPM fits.

In [ ]:
BOOT_DRAWS = 1000        # K1 concept-cluster bootstrap draws for the decomposition CIs   (original: 1000)
RAND_T_DRAWS = 2000      # INFERENCE randomization-t / Freedman-Lane draws                  (original: 2000)
WILD_REPS = 9999         # Webb 6-point wild cluster score bootstrap draws                  (original: 9999)
OUT_DIR = Path("results_demo")   # where run_k1 / run_inference write their CSV / JSON (original: results/)

## The entry point: `eval.py`

`eval.py` runs each stage as a separate script, so a broken stage is reported as NOT RUN without touching the others.
The stage table below is copied from `eval.py`. The demo covers the estimation core of the **run** stage
(`k/k_run.py --stage all`). The freeze stage is represented by the frozen spec values carried in the data file (MDE80s, seeds, draw counts).

In [ ]:
STAGES = {"freeze": ["k/k_freeze.py", "--reps", "300"], "smoke": ["k/k_run.py", "--synthetic"],
          "run": ["k/k_run.py", "--stage", "all"], "audit": ["audit/audit_k.py"], "report": ["k/report.py"],
          "placebo": ["audit/audit_placebo.py"]}

for st, cmd in STAGES.items():
    print(f"{st:8s} -> {' '.join(cmd)}")

## Vendored estimator: `d2/src/ppml.py` (byte-exact, unchanged)

This is Poisson pseudo-maximum likelihood (PPML) with high-dimensional fixed effects. It uses IRLS, and each IRLS step
projects out the FE space exactly with a weighted sparse LU. Standard errors are CRV1, clustered by concept, with
pyfixest's small-sample factor. Before fitting, it iteratively prunes singleton FE levels and FE levels whose outcome is all 0.
The last line only wraps the functions in a `ppml` namespace, so the downstream code can call `ppml.fit(...)` as in the original.

In [ ]:
def _prune(y: np.ndarray, fes: list[np.ndarray]) -> np.ndarray:
    """Iteratively drop singleton FE levels and FE levels with all-zero outcome (separation)."""
    keep = np.ones(len(y), bool)
    for _ in range(50):
        changed = False
        for f in fes:
            cnt = np.bincount(f[keep], minlength=f.max() + 1)
            sy = np.bincount(f[keep], weights=y[keep], minlength=f.max() + 1)
            bad = (cnt <= 1) | (sy <= 0)
            drop = keep & bad[f]
            if drop.any():
                keep &= ~drop
                changed = True
        if not changed:
            break
    return keep


def _demean(M: np.ndarray, w: np.ndarray, fes: list[np.ndarray]) -> np.ndarray:
    """Exact weighted projection off the FE space: solve (D'WD + eps I) c = D'W M with a sparse LU (D = [D1 D2])."""
    from scipy import sparse
    from scipy.sparse.linalg import splu
    n = len(w)
    rows, cols, off = [], [], 0
    for f in fes:
        rows.append(np.arange(n))
        cols.append(f + off)
        off += int(f.max()) + 1
    D = sparse.csc_matrix((np.ones(n * len(fes)), (np.concatenate(rows), np.concatenate(cols))), shape=(n, off))
    DW = sparse.csc_matrix(D.multiply(w[:, None]))
    A = sparse.csc_matrix(D.T @ DW) + sparse.identity(off, format="csc") * 1e-9
    c = splu(A).solve(np.asarray(DW.T @ M))
    return M - D @ c


def pd_nunique_per_level(f: np.ndarray, g: np.ndarray) -> int:
    """Max number of distinct clusters within any FE level."""
    pairs = np.unique(np.stack([f, g], 1), axis=0)
    return int(np.bincount(pairs[:, 0]).max())


def fit(y: np.ndarray, X: np.ndarray, fes: list[np.ndarray], offset: np.ndarray, cluster: np.ndarray,
        maxit: int = 100, tol: float = 1e-9) -> dict | None:
    keep = _prune(y.astype(float), fes)
    if keep.sum() < X.shape[1] + 5:
        return None
    y, X, off, cl = y[keep].astype(float), X[keep], offset[keep], cluster[keep]
    fes = [np.unique(f[keep], return_inverse=True)[1] for f in fes]
    if np.linalg.matrix_rank(_demean(X, np.ones(len(y)), fes)) < X.shape[1]:
        return None
    mu = np.maximum(y, 0.1 * y.mean() + 1e-3)
    eta = np.log(mu)
    beta = np.zeros(X.shape[1])
    dev_old = np.inf
    for _ in range(maxit):
        z = eta - off + (y - mu) / mu
        M = _demean(np.column_stack([z, X]), mu, fes)
        zt, Xt = M[:, 0], M[:, 1:]
        WX = Xt * mu[:, None]
        beta = np.linalg.solve(Xt.T @ WX, WX.T @ zt)
        resid = zt - Xt @ beta
        eta = z - resid + off  # eta = X b + FE + offset
        eta = np.clip(eta, -30, 30)
        mu = np.exp(eta)
        dev = 2 * np.sum(np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0) - (y - mu))
        if abs(dev - dev_old) / (abs(dev) + 0.1) < tol:
            break
        dev_old = dev
    Xt = _demean(X, mu, fes)
    H = Xt.T @ (Xt * mu[:, None])
    Hi = np.linalg.inv(H)
    sc = Xt * (y - mu)[:, None]
    _, g = np.unique(cl, return_inverse=True)
    G = g.max() + 1
    S = np.zeros((G, X.shape[1]))
    np.add.at(S, g, sc)
    n = len(y)
    # small-sample factor as pyfixest CRV1 (fixef_k='nested'): FE levels nested in clusters are not counted
    k_fe = 0
    for f in fes:
        nested = np.all(np.bincount(f, minlength=f.max() + 1) == 0) or (
            pd_nunique_per_level(f, g) <= 1)
        if not nested:
            k_fe += int(f.max()) + 1 - 1
    k = X.shape[1] + k_fe + (1 if k_fe else 0)
    adj = G / (G - 1) * (n - 1) / (n - k) if G > 1 and n > k else 1.0
    V = adj * Hi @ (S.T @ S) @ Hi
    return {"coef": beta, "se": np.sqrt(np.diag(V)), "n": int(n), "G": int(G), "mu": mu, "keep": keep,
            "Xt": Xt, "y": y, "cl": g}


ppml = types.SimpleNamespace(_prune=_prune, _demean=_demean, pd_nunique_per_level=pd_nunique_per_level, fit=fit)

## Vendored sample definition: `d2/src/models.py` → `primary_sample`

This is the frozen analysis sample: main arm, the fold, the `kw5` window, the `MAIN` population, and complete
regressors. `primary_sample` also builds the fixed-effect codes. The co-primary specification used throughout K1 and
INFERENCE is **concept + entry year `e` + host subfield `d`** (`cfe`, `efe`, `dfe`). `cxe`/`dxe` (concept×year,
subfield×year) are the stricter primary FE, which appear only as a side row. The offset is `log(n_entry_papers)`.

In [ ]:
A_VAR = "A_cont"  # primary anchoring measure after the pre-declared nativeness fallback (deviations.md #1)


CT_VAR = "CT"


EVENT_CONTROLS = ["prox_od", "RD", "log_n_partner_tags", "cov", "demic", "mean_topic_score", "boundary_share",
                  "abstract_share"]


SECONDARY_EXTRA = ["mom_d", "log_centrality", "log_W1"]


def primary_sample(df: pd.DataFrame, pop: str = "MAIN", fold: str = "screen") -> pd.DataFrame:
    s = df[(df.arm == "main") & (df.fold == fold) & df.kw5 & df[pop]].copy()
    need = [A_VAR, CT_VAR] + EVENT_CONTROLS + SECONDARY_EXTRA
    s = s.dropna(subset=[c for c in need if c in s.columns])
    s["cxe"] = pd.factorize(s.concept_id + "_" + s.e.astype(str))[0]
    s["dxe"] = pd.factorize(s.d.astype(str) + "_" + s.e.astype(str))[0]
    s["cfe"] = pd.factorize(s.concept_id)[0]
    s["efe"] = pd.factorize(s.e.astype(str))[0]
    s["dfe"] = pd.factorize(s.d.astype(str))[0]
    s["cx2"] = pd.factorize(s.concept_id + "_" + ((s.e - 2005) // 2).astype(str))[0]
    s["offset"] = np.log(s.n_entry_papers.astype(float))
    return s.reset_index(drop=True)


models = types.SimpleNamespace(A_VAR=A_VAR, CT_VAR=CT_VAR, EVENT_CONTROLS=EVENT_CONTROLS,
                               SECONDARY_EXTRA=SECONDARY_EXTRA, primary_sample=primary_sample)

## `k/k_lib.py` (1/4): constants, utilities, fold loader

These are the original constants and helpers. There are two notebook edits:
* `load_fold` builds the held-out table from the loaded JSON (one example per concept) instead of reading the parquet
  file. The frozen `models.primary_sample` filter is then applied exactly as before.
* `rel` makes output paths relative to the working directory (the original walked up to the run root). `controls` keeps only its non-MeSH branch.

`add_outcomes` defines the K1 outcomes: `Any = 1[Y>=1]` (extensive margin), the ladder cut-offs `Y>=3` and `Y>=5`,
`EST_bin`, and `log_nep`, the log entry-paper count used as a regressor in the LPMs.

In [ ]:
WS = Path.cwd()  # original: Path(__file__).resolve().parents[1]

SEED = 20261001


CTRL2 = models.EVENT_CONTROLS + models.SECONDARY_EXTRA


FOLDS = ("SCREEN", "HELDOUT", "MESH")


FIELD_OF_GROUP = {31: "Physics/Astro", 17: "CS"}


GROUPS = ("Physics/Astro", "CS", "other")


def dump(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=1, default=_json_default))


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return str(o)


def clean(x):
    """Recursively replace NaN / inf with None so the JSON is strict."""
    if isinstance(x, dict):
        return {k: clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [clean(v) for v in x]
    if isinstance(x, (float, np.floating)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.bool_):
        return bool(x)
    return x


def rel(p: Path | str) -> str:
    """Path relative to the run root (…/run_xxx) for source columns; never an absolute server path."""
    p = Path(p).resolve()
    root = WS  # notebook: relative to the working directory
    try:
        return str(p.relative_to(root))
    except ValueError:
        return str(p)


def add_outcomes(s: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()
    y = s["Y_strict"].astype(float)
    s["Any"] = (y >= 1).astype(float)
    s["Y_ge3"] = (y >= 3).astype(float)
    s["Y_ge5"] = (y >= 5).astype(float)
    s["EST_bin"] = s["EST_bin"].astype(float)
    s["log_nep"] = np.log(s["n_entry_papers"].astype(float))
    return s


def load_fold(fold: str) -> pd.DataFrame:
    """Frozen per-event sample of a fold with the co-primary FE columns (cfe/efe/dfe, cxe/dxe) and offset.
    SCREEN/HELDOUT: models.primary_sample (arm main, fold, kw5, MAIN, complete A_cont/CT/CTRL2) exactly as
    audit/audit_perm.py filters. MESH: models_mesh.sample(outcomes_mesh, [A_cont, CT] + secondary_controls).
    NOTEBOOK: only HELDOUT is shipped; its rows come from mini_demo_data.json (one example = one concept)."""
    if fold == "HELDOUT":
        raw = pd.DataFrame([{"concept_id": ex["concept_id"], **row} for ex in examples for row in ex["entries"]])
        s = models.primary_sample(raw, fold="heldout")
    else:
        raise ValueError(f"{fold}: not shipped with the demo (only HELDOUT)")
    s = add_outcomes(s)
    s["fold"] = fold
    return s.reset_index(drop=True)


def controls(fold: str) -> list[str]:
    return CTRL2  # original: mesh_controls() if fold == "MESH" else CTRL2  (MeSH fold not in the demo)


def fe_cols_for(fe: str) -> list[str]:
    return {"coprimary": ["cfe", "efe", "dfe"], "primary": ["cxe", "dxe"], "k3": ["cfe", "fxe", "dfe"]}[fe]

## `k/k_lib.py` (2/4): pruning and the cached-projection LPM (extensive margin)

`LPM` is OLS with the FE projected out exactly. The singleton-pruned sample depends only on the FE structure, so the
projector is factorised **once** and reused for every outcome and every permuted regressor. This is what makes 2,000
randomization draws cheap. `lpm_row` reports the effect of one SD of `A_cont` in percentage points (pp/SD) with
CRV1 t(G−1) CIs, and relative to the base rate P(Y≥1).

In [ ]:
def prune_singletons(fes: list[np.ndarray]) -> np.ndarray:
    """Iterated singleton pruning (LPM / binary models: no separation rule)."""
    keep = np.ones(len(fes[0]), bool)
    for _ in range(100):
        changed = False
        for f in fes:
            cnt = np.bincount(f[keep], minlength=f.max() + 1)
            drop = keep & (cnt[f] <= 1)
            if drop.any():
                keep &= ~drop
                changed = True
        if not changed:
            break
    return keep


def prune_binary_separation(y: np.ndarray, fes: list[np.ndarray]) -> np.ndarray:
    """Iterated pruning for FE-logit: singleton levels and levels whose outcome is all-0 or all-1."""
    keep = np.ones(len(y), bool)
    for _ in range(100):
        changed = False
        for f in fes:
            cnt = np.bincount(f[keep], minlength=f.max() + 1)
            sy = np.bincount(f[keep], weights=y[keep], minlength=f.max() + 1)
            bad = (cnt <= 1) | (sy <= 0) | (sy >= cnt)
            drop = keep & bad[f]
            if drop.any():
                keep &= ~drop
                changed = True
        if not changed:
            break
    return keep


def _relabel(fes: list[np.ndarray], keep: np.ndarray) -> list[np.ndarray]:
    return [np.unique(f[keep], return_inverse=True)[1] for f in fes]


def _dummy(fes: list[np.ndarray]) -> sparse.csc_matrix:
    n = len(fes[0])
    rows, cols, off = [], [], 0
    for f in fes:
        rows.append(np.arange(n))
        cols.append(f + off)
        off += int(f.max()) + 1
    return sparse.csc_matrix((np.ones(n * len(fes)), (np.concatenate(rows), np.concatenate(cols))), shape=(n, off))


def ssc_adj(n: int, kx: int, fes: list[np.ndarray], g: np.ndarray) -> float:
    """pyfixest CRV1 small-sample factor with fixef_k='nested' (identical rule to vendored ppml.fit)."""
    G = int(g.max()) + 1
    k_fe = 0
    for f in fes:
        if not (ppml.pd_nunique_per_level(f, g) <= 1):
            k_fe += int(f.max())
    k = kx + k_fe + (1 if k_fe else 0)
    return G / (G - 1) * (n - 1) / (n - k) if G > 1 and n > k else 1.0


class LPM:
    """OLS with HDFE by exact projection. The retained sample (iterated singleton pruning) depends only on the FE
    structure, so the projector is factorised ONCE and reused for every outcome / permuted regressor."""

    def __init__(self, fes: list[np.ndarray], cluster: np.ndarray):
        self.keep = prune_singletons(fes)
        self.fes = _relabel(fes, self.keep)
        self.D = _dummy(self.fes)
        A = sparse.csc_matrix(self.D.T @ self.D) + sparse.identity(self.D.shape[1], format="csc") * 1e-9
        self.lu = splu(A)
        self.g = np.unique(cluster[self.keep], return_inverse=True)[1]
        self.G = int(self.g.max()) + 1
        self.n = int(self.keep.sum())

    def __getstate__(self):  # SuperLU is not picklable: refactorise in the worker
        st = dict(self.__dict__)
        st.pop("lu", None)
        return st

    def __setstate__(self, st):
        self.__dict__.update(st)
        A = sparse.csc_matrix(self.D.T @ self.D) + sparse.identity(self.D.shape[1], format="csc") * 1e-9
        self.lu = splu(A)

    def demean(self, M: np.ndarray) -> np.ndarray:
        M = np.asarray(M, float)
        return M - self.D @ self.lu.solve(np.asarray(self.D.T @ M))

    def fit(self, y: np.ndarray, X: np.ndarray, Xt: np.ndarray | None = None) -> dict:
        """y, X on the FULL input sample (keep applied here); Xt optional pre-demeaned X on the kept sample."""
        yk = np.asarray(y, float)[self.keep]
        if Xt is None:
            Xt = self.demean(np.asarray(X, float)[self.keep])
        yt = self.demean(yk[:, None])[:, 0]
        XtX = Xt.T @ Xt
        beta = np.linalg.solve(XtX, Xt.T @ yt)
        e = yt - Xt @ beta
        Hi = np.linalg.inv(XtX)
        S = np.zeros((self.G, Xt.shape[1]))
        np.add.at(S, self.g, Xt * e[:, None])
        adj = ssc_adj(self.n, Xt.shape[1], self.fes, self.g)
        V = adj * Hi @ (S.T @ S) @ Hi
        return {"coef": beta, "se": np.sqrt(np.diag(V)), "V": V, "n": self.n, "G": self.G, "resid": e, "Xt": Xt,
                "yt": yt}


def lpm_row(s: pd.DataFrame, y: str, xvars: list[str], fe: str, target: str = "A_cont") -> dict:
    fes = [s[c].to_numpy() for c in fe_cols_for(fe)]
    eng = LPM(fes, s.concept_id.to_numpy())
    r = eng.fit(s[y].to_numpy(float), s[xvars].to_numpy(float))
    j = xvars.index(target)
    b, se = float(r["coef"][j]), float(r["se"][j])
    sd = float(s.loc[eng.keep, target].std())
    G = r["G"]
    tq = stats.t.ppf(0.975, G - 1)
    base = float(s.loc[eng.keep, y].mean())
    return {"model": "LPM", "y": y, "fe": fe, "b": b, "se": se, "t": b / se, "p_crv1": float(2 * stats.t.sf(abs(b / se), G - 1)),
            "sd": sd, "pp_per_sd": 100 * b * sd, "ci_pp_per_sd": [100 * (b - tq * se) * sd, 100 * (b + tq * se) * sd],
            "base_rate": base, "rel_to_base": (b * sd) / base if base > 0 else None,
            "N": r["n"], "N_input": int(len(s)), "G": G, "retained_share": r["n"] / len(s)}

## `k/k_lib.py` (3/4): FE-logit and PPML wrappers

* `logit_fe_fit` / `logit_row`: an unconditional FE logit (IRLS with a weighted within-transformation) on `1[Y>=1]`,
  reported as OR/SD. This is a corroborating extensive-margin row.
* `ppml_fit` / `ppml_row`: thin wrappers around the vendored `ppml.fit`, reported as IRR/SD. `A=` swaps in a permuted
  `A_cont` for the randomization draws. With `offset=False` on the binary `Any`, this gives the log-link P(Y≥1) ratio
  used by the decomposition.

In [ ]:
def logit_fe_fit(y: np.ndarray, X: np.ndarray, fes: list[np.ndarray], cluster: np.ndarray, maxit: int = 200,
                 tol: float = 1e-10) -> dict | None:
    """Unconditional FE logit by IRLS with a weighted within-transformation (exact sparse projection, as vendored
    ppml.fit does for the log link). FE levels with all-0 / all-1 outcomes and singletons are pruned (iterated)."""
    keep = prune_binary_separation(y.astype(float), fes)
    if keep.sum() < X.shape[1] + 5:
        return None
    yk, Xk, cl = y[keep].astype(float), X[keep].astype(float), cluster[keep]
    fk = _relabel(fes, keep)
    p = np.clip(0.5 * np.ones_like(yk) * 0 + (yk + 0.5) / 2, 0.05, 0.95)
    eta = np.log(p / (1 - p))
    dev_old = np.inf
    beta = np.zeros(Xk.shape[1])
    for _ in range(maxit):
        w = p * (1 - p)
        z = eta + (yk - p) / w
        M = ppml._demean(np.column_stack([z, Xk]), w, fk)
        zt, Xt = M[:, 0], M[:, 1:]
        WX = Xt * w[:, None]
        beta = np.linalg.solve(Xt.T @ WX, WX.T @ zt)
        resid = zt - Xt @ beta
        eta = np.clip(z - resid, -30, 30)
        p = 1 / (1 + np.exp(-eta))
        p = np.clip(p, 1e-12, 1 - 1e-12)
        dev = -2 * np.sum(yk * np.log(p) + (1 - yk) * np.log(1 - p))
        if abs(dev - dev_old) / (abs(dev) + 0.1) < tol:
            break
        dev_old = dev
    w = p * (1 - p)
    Xt = ppml._demean(Xk, w, fk)
    Hi = np.linalg.inv(Xt.T @ (Xt * w[:, None]))
    g = np.unique(cl, return_inverse=True)[1]
    G = int(g.max()) + 1
    S = np.zeros((G, Xk.shape[1]))
    np.add.at(S, g, Xt * (yk - p)[:, None])
    adj = ssc_adj(len(yk), Xk.shape[1], fk, g)
    V = adj * Hi @ (S.T @ S) @ Hi
    return {"coef": beta, "se": np.sqrt(np.diag(V)), "n": int(len(yk)), "G": G, "keep": keep}


def logit_row(s: pd.DataFrame, y: str, xvars: list[str], fe: str, target: str = "A_cont") -> dict:
    fes = [s[c].to_numpy() for c in fe_cols_for(fe)]
    r = logit_fe_fit(s[y].to_numpy(float), s[xvars].to_numpy(float), fes, s.concept_id.to_numpy())
    if r is None:
        return {"model": "FE-logit", "y": y, "fe": fe, "status": "not estimable (too few events after pruning)"}
    j = xvars.index(target)
    b, se = float(r["coef"][j]), float(r["se"][j])
    sd = float(s.loc[r["keep"], target].std())
    tq = stats.t.ppf(0.975, r["G"] - 1)
    conc_all = s.concept_id.nunique()
    conc_kept = s.loc[r["keep"], "concept_id"].nunique()
    return {"model": "FE-logit", "y": y, "fe": fe, "b": b, "se": se, "t": b / se,
            "p_crv1": float(2 * stats.t.sf(abs(b / se), r["G"] - 1)), "sd": sd, "or_per_sd": math.exp(b * sd),
            "ci_or_per_sd": [math.exp((b - tq * se) * sd), math.exp((b + tq * se) * sd)], "N": r["n"],
            "N_input": int(len(s)), "G": r["G"], "concepts_dropped_all0_all1_or_singleton": int(conc_all - conc_kept),
            "retained_share": r["n"] / len(s), "status": "ok"}


def ppml_fit(s: pd.DataFrame, y: str, xvars: list[str], fe: str, offset: bool = True, A: np.ndarray | None = None,
             target: str = "A_cont", yv: np.ndarray | None = None) -> dict | None:
    """Vendored ppml.fit (unchanged). A overrides the target column (randomization draws); yv overrides y."""
    X = s[xvars].to_numpy(float)
    j = xvars.index(target)
    if A is not None:
        X = X.copy()
        X[:, j] = A
    fes = [s[c].to_numpy() for c in fe_cols_for(fe)]
    yy = s[y].to_numpy(float) if yv is None else np.asarray(yv, float)
    off = s["offset"].to_numpy() if offset else np.zeros(len(s))
    try:
        r = ppml.fit(yy, X, fes, off, s.concept_id.to_numpy(), maxit=500, tol=1e-12)
    except (np.linalg.LinAlgError, ValueError, RuntimeError, FloatingPointError):
        return None
    if r is None:
        return None
    b, se = float(r["coef"][j]), float(r["se"][j])
    if not (np.isfinite(b) and np.isfinite(se)) or se <= 0:
        return None
    return {"b": b, "se": se, "t": b / se, "n": r["n"], "G": r["G"], "keep": r["keep"], "coef": r["coef"],
            "se_all": r["se"], "mu": r["mu"], "Xt": r["Xt"], "y": r["y"], "cl": r["cl"]}


def ppml_row(s: pd.DataFrame, y: str, xvars: list[str], fe: str, offset: bool = True, target: str = "A_cont",
             label: str = "") -> dict:
    r = ppml_fit(s, y, xvars, fe, offset=offset, target=target)
    if r is None:
        return {"model": "PPML", "y": y, "fe": fe, "status": "not estimable", "label": label}
    sd = float(s.loc[r["keep"], target].std())
    G = r["G"]
    tq = stats.t.ppf(0.975, G - 1)
    b, se = r["b"], r["se"]
    return {"model": "PPML", "y": y, "fe": fe, "offset": offset, "b": b, "se": se, "t": b / se,
            "p_crv1": float(2 * stats.t.sf(abs(b / se), G - 1)), "sd": sd, "irr_per_sd": math.exp(b * sd),
            "ci_irr_per_sd": [math.exp((b - tq * se) * sd), math.exp((b + tq * se) * sd)], "N": r["n"],
            "N_input": int(len(s)), "G": G, "retained_share": r["n"] / len(s), "status": "ok", "label": label}

## `k/k_lib.py` (4/4): wild bootstrap, randomization engines, IVW, cluster bootstrap

* **Wild cluster score bootstrap** (Kline-Santos): cluster scores come from the *restricted* fit (b_A = 0). The
  statistic T = (Σ s_g)² / Σ s_g² is re-drawn with Webb 6-point or Rademacher cluster weights.
* **Randomization-t** (the declared headline p): permute `A_cont` *within concept*, refit, and compare |t*| with |t_obs|.
  **Freedman-Lane** permutes the residual of `A_cont` given the controls and FE instead.
* **IVW**: fixed-effect inverse-variance pooling across folds, with Q / I² (and DerSimonian-Laird if I² > 0.5).
* **cluster_resample**: a concept-level bootstrap. Each drawn copy becomes a distinct concept.

The final line collects the `k_lib` definitions into a `K` namespace, so the `k_run.py` code below can call `K.xxx` as it does with `import k_lib as K`.

In [ ]:
WEBB = np.array([-math.sqrt(1.5), -1.0, -math.sqrt(0.5), math.sqrt(0.5), 1.0, math.sqrt(1.5)])


def score_components_ppml(s: pd.DataFrame, y: str, xvars: list[str], fe: str, target: str = "A_cont",
                          offset: bool = True) -> np.ndarray | None:
    """Cluster scores of the target under the restricted (b_target = 0) PPML fit, target partialled on the
    restricted design with the restricted weights -- identical construction to vendored ppml.wild_score_test."""
    j = xvars.index(target)
    X = s[xvars].to_numpy(float)
    Xr, x2 = np.delete(X, j, axis=1), X[:, j]
    fes = [s[c].to_numpy() for c in fe_cols_for(fe)]
    off = s["offset"].to_numpy() if offset else np.zeros(len(s))
    r = ppml.fit(s[y].to_numpy(float), Xr, fes, off, s.concept_id.to_numpy())
    if r is None:
        return None
    keep, mu = r["keep"], r["mu"]
    fk = _relabel(fes, keep)
    D = ppml._demean(np.column_stack([x2[keep], Xr[keep]]), mu, fk)
    x2t, Xrt = D[:, 0], D[:, 1:]
    b = np.linalg.solve(Xrt.T @ (Xrt * mu[:, None]), (Xrt * mu[:, None]).T @ x2t)
    x2r = x2t - Xrt @ b
    return np.bincount(r["cl"], weights=x2r * (r["y"] - mu))


def score_components_lpm(s: pd.DataFrame, y: str, xvars: list[str], fe: str, target: str = "A_cont") -> np.ndarray:
    """Linear analogue: restricted OLS residuals times the target partialled on the restricted design (HDFE)."""
    j = xvars.index(target)
    fes = [s[c].to_numpy() for c in fe_cols_for(fe)]
    eng = LPM(fes, s.concept_id.to_numpy())
    X = s[xvars].to_numpy(float)[eng.keep]
    M = eng.demean(np.column_stack([s[y].to_numpy(float)[eng.keep], X]))
    yt, Xt = M[:, 0], M[:, 1:]
    Xrt, x2t = np.delete(Xt, j, axis=1), Xt[:, j]
    br = np.linalg.lstsq(Xrt, yt, rcond=None)[0]
    er = yt - Xrt @ br
    x2r = x2t - Xrt @ np.linalg.lstsq(Xrt, x2t, rcond=None)[0]
    return np.bincount(eng.g, weights=x2r * er)


def wild_score_test_webb(sc: np.ndarray | None, rng: np.random.Generator, reps: int = 9999,
                         weights: str = "webb", chunk: int = 2000) -> dict | None:
    """Kline-Santos score bootstrap statistic T = (sum s_g)^2 / sum s_g^2 with Webb 6-point (default) or Rademacher
    cluster weights. Mirrors ppml.wild_score_test; returns p and the number of clusters."""
    if sc is None or len(sc) < 2 or (sc ** 2).sum() == 0:
        return None
    T = sc.sum() ** 2 / (sc ** 2).sum()
    ge = 0
    done = 0
    while done < reps:
        m = min(chunk, reps - done)
        if weights == "webb":
            eps = WEBB[rng.integers(0, 6, size=(m, len(sc)))]
        else:
            eps = rng.choice([-1.0, 1.0], size=(m, len(sc)))
        Ts = (eps @ sc) ** 2 / (sc ** 2).sum()
        ge += int((Ts >= T).sum())
        done += m
    return {"p": float((1 + ge) / (reps + 1)), "T": float(T), "G": int(len(sc)), "reps": reps, "weights": weights}


def within_concept_perm(values: np.ndarray, groups: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Permute values within each group (single-member groups are unchanged). Vectorised: sort by (group, random)."""
    r = rng.random(len(values))
    order = np.lexsort((r, groups))          # positions sorted by group then random key
    base = np.lexsort((np.arange(len(values)), groups))  # positions sorted by group, stable
    out = np.empty_like(values)
    out[base] = values[order]
    return out


def freedman_lane_parts(s: pd.DataFrame, xvars: list[str], fe: str, target: str = "A_cont") -> tuple[np.ndarray, np.ndarray]:
    """Residualise the target on the other regressors + FE by OLS (same FE, singleton-pruned projector);
    rows dropped by singleton pruning keep their observed value as the 'fitted' part with zero residual."""
    fes = [s[c].to_numpy() for c in fe_cols_for(fe)]
    eng = LPM(fes, s.concept_id.to_numpy())
    X = s[xvars].to_numpy(float)
    j = xvars.index(target)
    Xk = X[eng.keep]
    M = eng.demean(Xk)
    at, Zt = M[:, j], np.delete(M, j, axis=1)
    g = np.linalg.lstsq(Zt, at, rcond=None)[0]
    resid_k = at - Zt @ g
    fitted = X[:, j].copy()
    resid = np.zeros(len(s))
    fitted[eng.keep] = X[eng.keep, j] - resid_k
    resid[eng.keep] = resid_k
    return fitted, resid


def rand_p(t_obs: float, ts: np.ndarray) -> dict:
    ts = np.asarray([t for t in ts if t is not None and np.isfinite(t)], float)
    B = len(ts)
    p = (1 + int((np.abs(ts) >= abs(t_obs)).sum())) / (1 + B)
    return {"p": float(p), "draws_converged": B, "mc_se": float(math.sqrt(p * (1 - p) / max(B, 1)))}


def ivw(est: list[float], se: list[float]) -> dict:
    est, se = np.asarray(est, float), np.asarray(se, float)
    ok = np.isfinite(est) & np.isfinite(se) & (se > 0)
    est, se = est[ok], se[ok]
    w = 1 / se ** 2
    m = float((w * est).sum() / w.sum())
    sm = float(math.sqrt(1 / w.sum()))
    Q = float((w * (est - m) ** 2).sum())
    k = len(est)
    I2 = float(max(0.0, (Q - (k - 1)) / Q)) if Q > 0 else 0.0
    out = {"est": m, "se": sm, "ci": [m - 1.96 * sm, m + 1.96 * sm], "z": m / sm,
           "p": float(2 * stats.norm.sf(abs(m / sm))), "Q": Q, "Q_df": k - 1,
           "Q_p": float(stats.chi2.sf(Q, k - 1)) if k > 1 else None, "I2": I2, "k": k}
    if I2 > 0.5 and k > 1:
        tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
        wr = 1 / (se ** 2 + tau2)
        mr = float((wr * est).sum() / wr.sum())
        sr = float(math.sqrt(1 / wr.sum()))
        out["random_effects_DL"] = {"est": mr, "se": sr, "ci": [mr - 1.96 * sr, mr + 1.96 * sr], "tau2": tau2}
    return out


def cluster_resample(s: pd.DataFrame, rng: np.random.Generator, fe: str = "coprimary") -> pd.DataFrame:
    """Resample concepts with replacement; each drawn copy becomes a distinct concept (FE level and cluster)."""
    cons = s.concept_id.unique()
    draw = rng.choice(cons, size=len(cons), replace=True)
    idx = s.groupby("concept_id").indices
    parts, labels = [], []
    for k, c in enumerate(draw):
        ii = idx[c]
        parts.append(ii)
        labels.append(np.full(len(ii), k))
    ii = np.concatenate(parts)
    b = s.iloc[ii].copy()
    b["concept_id"] = np.char.add("b", np.concatenate(labels).astype(str))
    b["cfe"] = pd.factorize(b.concept_id)[0]
    b["cxe"] = pd.factorize(b.concept_id + "_" + b.e.astype(str))[0]
    return b.reset_index(drop=True)


K = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith("_") or k in ("_relabel", "_dummy")})

## `k/k_run.py`: regressor lists and spec helpers

* The **count** models (PPML) use `A_cont + CT + controls` with the `log(n_entry_papers)` offset.
* The **LPM** models add `log_nep` as a regressor instead.

`mde_of` reads the MDE80s that were simulated and frozen in the spec *before* any K coefficient was computed.

In [ ]:
LABEL = "POST-CONFIRMATION EXPLORATORY"


_W: dict = {}


def xv_lpm(fold: str) -> list[str]:
    return ["A_cont", "CT"] + K.controls(fold) + ["log_nep"]


def xv_cnt(fold: str) -> list[str]:
    return ["A_cont", "CT"] + K.controls(fold)


def mde_of(spec: dict, kind: str, fold: str):
    m = spec["MDE"]
    if kind == "ext":
        return m["extensive_pp_per_sd"].get(fold, {}).get("MDE80")
    if kind == "int":
        return m["intensive_irr_per_sd"].get(fold, {}).get("MDE80")
    return None

## K1: which margin does the effect act on?

`run_k1` is the original function. For each fold it fits the rows below:

| Row | Model | Reads as |
|---|---|---|
| `K1_total` | PPML on `Y_strict` | the co-primary total effect, IRR/SD (gate R0b: held-out 1.187) |
| `K1a_LPM` | LPM on `Any = 1[Y>=1]` | **extensive** margin, pp/SD (primary) |
| `K1a_logit`, `K1a_loglink` | FE-logit / log-link PPML on `Any` | corroborating extensive rows |
| `K1b` | PPML on `Y_strict` for entries with `Y>=1` | **intensive** margin (conditional, descriptive) |
| `K1d_ladder` | LPM on `Y>=1`, `Y>=3`, `Y>=5`, `EST_bin` | where along the distribution the effect sits |
| `K1e_side_primaryFE` | stricter concept×year FE | an underpowered side row |

The log-link decomposition splits the total log effect into extensive + intensive parts, with an extensive
**share** = b_ext / (b_ext + b_int). Its CIs come from `BOOT_DRAWS` concept-cluster bootstrap draws (`_boot_draw`).
Only the held-out fold is shipped, so the "IVW" rows here pool a single fold and equal the held-out estimates.
The real 3-fold IVW is shown from the stored results at the end. The only edit is the process pool → loop.

In [ ]:
def _boot_draw(args: tuple) -> dict:
    fold, i = args
    s = _W["samples"][fold]
    rng = np.random.default_rng(K.SEED + 3_000_000 + i)
    b = K.cluster_resample(s, rng)
    out = {"fold": fold, "draw": i}
    rt = K.ppml_fit(b, "Y_strict", xv_cnt(fold), "coprimary")
    re = K.ppml_fit(b, "Any", xv_lpm(fold), "coprimary", offset=False)
    sub = b[b.Y_strict >= 1].reset_index(drop=True)
    ri = K.ppml_fit(sub, "Y_strict", xv_cnt(fold), "coprimary")
    out["b_tot"] = None if rt is None else rt["b"]
    out["b_ext"] = None if re is None else re["b"]
    out["b_int"] = None if ri is None else ri["b"]
    return out


def run_k1(samples: dict, spec: dict, out: Path, boot: int) -> dict:
    rows, per = [], {}
    src = f"{K.rel(out)}/k1_rows.csv"
    for fold, s in samples.items():
        tot = K.ppml_row(s, "Y_strict", xv_cnt(fold), "coprimary")
        lpm = K.lpm_row(s, "Any", xv_lpm(fold), "coprimary")
        lgt = K.logit_row(s, "Any", xv_lpm(fold), "coprimary")
        ll = K.ppml_row(s, "Any", xv_lpm(fold), "coprimary", offset=False)
        sub = s[s.Y_strict >= 1].reset_index(drop=True)
        k1b = K.ppml_row(sub, "Y_strict", xv_cnt(fold), "coprimary",
                         label="conditional on uptake starting; descriptive, subject to selection")
        ladder = {y: K.lpm_row(s, y, xv_lpm(fold), "coprimary") for y in ("Any", "Y_ge3", "Y_ge5", "EST_bin")}
        side_lpm = K.lpm_row(s, "Any", xv_lpm(fold), "primary")
        side_k1b = K.ppml_row(sub, "Y_strict", xv_cnt(fold), "primary")
        per[fold] = {"total": tot, "K1a_LPM": lpm, "K1a_logit": lgt, "K1a_loglink": ll, "K1b": k1b, "ladder": ladder,
                     "side_primary_LPM": side_lpm, "side_primary_K1b": side_k1b}

        def add(test, model, fe, outcome, est, unit, lo, hi, r, base=None, mde=None, note=""):
            rows.append({"test": test, "fold": fold, "model": model, "fe": fe, "outcome": outcome, "estimate": est,
                         "unit": unit, "ci_lo": lo, "ci_hi": hi, "p_crv1": r.get("p_crv1"), "N": r.get("N"),
                         "G": r.get("G"), "base_rate": base, "mde80": mde, "b": r.get("b"), "se": r.get("se"),
                         "sd_A": r.get("sd"), "label": LABEL, "note": note, "source": src})

        add("K1_total", "PPML", "concept+e+d", "Y_strict", tot["irr_per_sd"], "IRR/SD", *tot["ci_irr_per_sd"], tot,
            note="co-primary total (= R0b)")
        add("K1a_LPM", "LPM", "concept+e+d", "Any", lpm["pp_per_sd"], "pp/SD", *lpm["ci_pp_per_sd"], lpm,
            base=lpm["base_rate"], mde=mde_of(spec, "ext", fold),
            note=f"relative to base rate: {lpm['rel_to_base']:.4f}; log(n_entry_papers) as regressor")
        if lgt.get("status") == "ok":
            add("K1a_logit", "FE-logit", "concept+e+d", "Any", lgt["or_per_sd"], "OR/SD", *lgt["ci_or_per_sd"], lgt,
                note=f"concepts dropped (all-0/all-1/singleton): {lgt['concepts_dropped_all0_all1_or_singleton']}")
        else:
            rows.append({"test": "K1a_logit", "fold": fold, "model": "FE-logit", "note": lgt.get("status"),
                         "label": LABEL, "source": src})
        add("K1a_loglink", "PPML (log link, binary)", "concept+e+d", "Any", ll["irr_per_sd"], "ratio of P(Y>=1)/SD",
            *ll["ci_irr_per_sd"], ll, base=lpm["base_rate"])
        add("K1b", "PPML on Y>=1", "concept+e+d", "Y_strict | Y>=1", k1b["irr_per_sd"], "IRR/SD",
            *k1b["ci_irr_per_sd"], k1b, mde=mde_of(spec, "int", fold),
            note="conditional on uptake starting; descriptive, subject to selection")
        for y, r in ladder.items():
            add("K1d_ladder", "LPM", "concept+e+d", y, r["pp_per_sd"], "pp/SD", *r["ci_pp_per_sd"], r,
                base=r["base_rate"], note=f"relative to base rate: {r['rel_to_base']}")
        add("K1e_side_primaryFE", "LPM", "concept x e + d x e", "Any", side_lpm["pp_per_sd"], "pp/SD",
            *side_lpm["ci_pp_per_sd"], side_lpm, base=side_lpm["base_rate"],
            note="underpowered (G < 50)" if side_lpm["G"] < 50 else "side row")
        if side_k1b.get("status") == "ok":
            add("K1e_side_primaryFE", "PPML on Y>=1", "concept x e + d x e", "Y_strict | Y>=1", side_k1b["irr_per_sd"],
                "IRR/SD", *side_k1b["ci_irr_per_sd"], side_k1b,
                note="underpowered (G < 50)" if side_k1b["G"] < 50 else "side row")
        else:
            rows.append({"test": "K1e_side_primaryFE", "fold": fold, "model": "PPML on Y>=1", "fe": "concept x e + d x e",
                         "note": "not estimable", "label": LABEL, "source": src})
    # ---------------- decomposition + cluster bootstrap
    t0 = time.time()
    res = []
    _W.update({"samples": samples})  # notebook: sequential loop instead of the spawn process pool
    for f in samples:
        for i in range(boot):
            res.append(_boot_draw((f, i)))
    bd = pd.DataFrame(res)
    bd.to_csv(out / "k1_bootstrap_draws.csv", index=False)
    logger.info(f"K1 bootstrap {len(bd)} draws in {time.time() - t0:.0f}s")
    dec = {"label": LABEL, "boot_draws_requested": boot, "folds": {}}
    for fold in samples:
        p = per[fold]
        sd = p["total"]["sd"]
        bt, be, bi = p["total"]["b"], p["K1a_loglink"]["b"], p["K1b"]["b"]
        d = bd[bd.fold == fold]
        ok = d.dropna(subset=["b_tot", "b_ext", "b_int"])
        sh = ok.b_ext / (ok.b_ext + ok.b_int)
        gap = ok.b_tot - (ok.b_ext + ok.b_int)
        pct = lambda v: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if len(v) else [None, None]  # noqa: E731
        dec["folds"][fold] = {
            "sd_full": sd, "b_total_sd": bt * sd, "b_ext_sd": be * sd, "b_int_sd": bi * sd,
            "ext_share": be / (be + bi), "gap_sd": (bt - (be + bi)) * sd,
            "ci_b_total_sd": [x * sd for x in pct(ok.b_tot)], "ci_b_ext_sd": [x * sd for x in pct(ok.b_ext)],
            "ci_b_int_sd": [x * sd for x in pct(ok.b_int)], "ci_ext_share": pct(sh), "ci_gap_sd": [x * sd for x in pct(gap)],
            "boot_se_ext_share": float(sh.std()), "boot_se_b_ext_sd": float(ok.b_ext.std() * sd),
            "boot_se_b_int_sd": float(ok.b_int.std() * sd),
            "draws_converged": int(len(ok)), "draws_failed": int(len(d) - len(ok)),
            "failed_share": float((len(d) - len(ok)) / max(len(d), 1)),
            "note": ("share conditions on converged draws" if (len(d) - len(ok)) > 0.05 * max(len(d), 1) else "")}
    # ---------------- IVW
    fl = list(samples)

    def iv(key, f_est, f_se):
        return K.ivw([f_est(per[f]) for f in fl], [f_se(per[f]) for f in fl])

    ivws = {
        "K1a_LPM_pp_per_sd": iv("lpm", lambda p: p["K1a_LPM"]["pp_per_sd"], lambda p: 100 * p["K1a_LPM"]["se"] * p["K1a_LPM"]["sd"]),
        "K1a_loglink_log_per_sd": iv("ll", lambda p: p["K1a_loglink"]["b"] * p["K1a_loglink"]["sd"],
                                     lambda p: p["K1a_loglink"]["se"] * p["K1a_loglink"]["sd"]),
        "K1b_log_irr_per_sd": iv("k1b", lambda p: p["K1b"]["b"] * p["K1b"]["sd"], lambda p: p["K1b"]["se"] * p["K1b"]["sd"]),
        "total_log_irr_per_sd": iv("tot", lambda p: p["total"]["b"] * p["total"]["sd"], lambda p: p["total"]["se"] * p["total"]["sd"]),
        "ext_share": K.ivw([dec["folds"][f]["ext_share"] for f in fl], [dec["folds"][f]["boot_se_ext_share"] for f in fl]),
    }
    if all(per[f]["K1a_logit"].get("status") == "ok" for f in fl):
        ivws["K1a_logit_log_or_per_sd"] = iv("lgt", lambda p: p["K1a_logit"]["b"] * p["K1a_logit"]["sd"],
                                             lambda p: p["K1a_logit"]["se"] * p["K1a_logit"]["sd"])
    ivws["folds_included"] = fl
    dec["ivw"] = ivws
    for key, unit, tr in [("K1a_LPM_pp_per_sd", "pp/SD", lambda x: x), ("K1a_loglink_log_per_sd", "ratio of P(Y>=1)/SD", math.exp),
                          ("K1b_log_irr_per_sd", "IRR/SD", math.exp), ("total_log_irr_per_sd", "IRR/SD", math.exp),
                          ("ext_share", "share", lambda x: x), ("K1a_logit_log_or_per_sd", "OR/SD", math.exp)]:
        if key not in ivws:
            continue
        v = ivws[key]
        rows.append({"test": f"K1f_IVW_{key}", "fold": "IVW", "model": "fixed-effect IVW", "fe": "concept+e+d",
                     "outcome": key, "estimate": tr(v["est"]), "unit": unit, "ci_lo": tr(v["ci"][0]),
                     "ci_hi": tr(v["ci"][1]), "p_crv1": v["p"], "N": None, "G": None, "base_rate": None,
                     "mde80": (spec["MDE"]["extensive_pp_per_sd"]["IVW"]["MDE80"] if key == "K1a_LPM_pp_per_sd" else
                               spec["MDE"]["intensive_irr_per_sd"]["IVW"]["MDE80"] if key == "K1b_log_irr_per_sd" else None),
                     "b": v["est"], "se": v["se"], "label": LABEL,
                     "note": f"Q={v['Q']:.3f} (df {v['Q_df']}, p {v['Q_p']}), I2={v['I2']:.3f}" +
                             (f"; DL random effects {v['random_effects_DL']}" if "random_effects_DL" in v else ""),
                     "source": f"{K.rel(out)}/k1_decomposition.json"})
    for fold in samples:
        d = dec["folds"][fold]
        rows.append({"test": "K1c_decomposition", "fold": fold, "model": "log-link two-part", "fe": "concept+e+d",
                     "outcome": "ext_share", "estimate": d["ext_share"], "unit": "share of (b_ext + b_int)",
                     "ci_lo": d["ci_ext_share"][0], "ci_hi": d["ci_ext_share"][1], "label": LABEL,
                     "note": f"b_total/SD {d['b_total_sd']:.4f}, b_ext/SD {d['b_ext_sd']:.4f}, b_int/SD {d['b_int_sd']:.4f}, "
                             f"gap/SD {d['gap_sd']:.4f}; bootstrap converged {d['draws_converged']}",
                     "source": f"{K.rel(out)}/k1_decomposition.json"})
    dec["per_fold_rows"] = per
    K.dump(K.clean(dec), out / "k1_decomposition.json")
    pd.DataFrame(rows).to_csv(out / "k1_rows.csv", index=False)
    logger.info(f"K1 IVW LPM {ivws['K1a_LPM_pp_per_sd']['est']:.3f} pp/SD {ivws['K1a_LPM_pp_per_sd']['ci']}; "
                f"K1b IRR {math.exp(ivws['K1b_log_irr_per_sd']['est']):.3f}")
    return dec

## INFERENCE: are the p values size-correct with few clusters?

The held-out fold has only G = 74 concept clusters in the count model. CRV1 p values from PPML can over-reject
with so few clusters. `run_inference` (the original function) therefore computes, for the co-primary count row and the K1a-LPM row:

* **randomization-t** (the headline): `A_cont` is permuted within concept for `RAND_T_DRAWS` draws. The seed is
  `SEED + draw`, the same as the original run, so the demo's draws are the first `RAND_T_DRAWS` of the original 2,000.
* **Freedman-Lane**: the permuted residual of `A_cont` given the controls and FE.
* **wild cluster score bootstrap**: Webb weights (`WILD_REPS` draws) and a 999-draw Rademacher check. These use the
  original seeds (fold index 1 = HELDOUT), so they reproduce the published values when `WILD_REPS` = 9999.
* **size diagnostics**: the share of null draws that CRV1 rejects at 5%, and the SD of the null t (1.0 when size is correct).

In [ ]:
def _inf_draw(i: int) -> dict:
    out = {"draw": i}
    for fold, d in _W["folds"].items():
        s = d["s"]
        A = d["A"]
        rng = np.random.default_rng(K.SEED + i)
        Ap = K.within_concept_perm(A, d["grp"], rng)
        rng2 = np.random.default_rng(K.SEED + 1_000_000 + i)
        Afl = d["fl_fit"] + K.within_concept_perm(d["fl_res"], d["grp"], rng2)
        Afl_l = d["fl_fit_l"] + K.within_concept_perm(d["fl_res_l"], d["grp"], rng2)
        for tag, Ax in (("plain", Ap), ("fl", Afl)):
            r = K.ppml_fit(s, "Y_strict", d["xv"], "coprimary", A=Ax)
            out[f"{fold}_cnt_{tag}_t"] = None if r is None else r["t"]
            out[f"{fold}_cnt_{tag}_b"] = None if r is None else r["b"]
            out[f"{fold}_cnt_{tag}_se"] = None if r is None else r["se"]
        eng: K.LPM = d["eng"]
        for tag, Ax in (("plain", Ap), ("fl", Afl_l)):
            Xt = d["Xt_l"].copy()
            Xt[:, 0] = eng.demean(Ax[eng.keep][:, None])[:, 0]
            yfull = s["Any"].to_numpy(float)
            r = eng.fit(yfull, None, Xt=Xt)
            out[f"{fold}_lpm_{tag}_t"] = float(r["coef"][0] / r["se"][0])
            out[f"{fold}_lpm_{tag}_b"] = float(r["coef"][0])
            out[f"{fold}_lpm_{tag}_se"] = float(r["se"][0])
    return out


def run_inference(samples: dict, out: Path, draws: int, wild_reps: int) -> dict:
    payload, obs = {"folds": {}}, {}
    for fold, s in samples.items():
        xv, xl = xv_cnt(fold), xv_lpm(fold)
        fes = [s[c].to_numpy() for c in K.fe_cols_for("coprimary")]
        eng = K.LPM(fes, s.concept_id.to_numpy())
        Xt_l = eng.demean(s[xl].to_numpy(float)[eng.keep])
        fl_fit, fl_res = K.freedman_lane_parts(s, xv, "coprimary")
        fl_fit_l, fl_res_l = K.freedman_lane_parts(s, xl, "coprimary")
        payload["folds"][fold] = {"s": s, "A": s.A_cont.to_numpy(float), "grp": pd.factorize(s.concept_id)[0],
                                  "xv": xv, "eng": eng, "Xt_l": Xt_l, "fl_fit": fl_fit, "fl_res": fl_res,
                                  "fl_fit_l": fl_fit_l, "fl_res_l": fl_res_l}
        rc = K.ppml_fit(s, "Y_strict", xv, "coprimary")
        rl = eng.fit(s["Any"].to_numpy(float), None, Xt=Xt_l)
        sd_c = float(s.loc[rc["keep"], "A_cont"].std())
        sd_l = float(s.loc[eng.keep, "A_cont"].std())
        obs[fold] = {"cnt": {"t": rc["t"], "b": rc["b"], "se": rc["se"], "G": rc["G"], "sd": sd_c,
                             "p_crv1": float(2 * stats.t.sf(abs(rc["t"]), rc["G"] - 1))},
                     "lpm": {"t": float(rl["coef"][0] / rl["se"][0]), "b": float(rl["coef"][0]), "se": float(rl["se"][0]),
                             "G": rl["G"], "sd": sd_l, "p_crv1": float(2 * stats.t.sf(abs(rl["coef"][0] / rl["se"][0]), rl["G"] - 1))}}
        # wild cluster score bootstraps (restricted fit; Webb 9999 headline, Rademacher 999 check)
        scc = K.score_components_ppml(s, "Y_strict", xv, "coprimary")
        scl = K.score_components_lpm(s, "Any", xl, "coprimary")
        k = K.FOLDS.index(fold)
        obs[fold]["cnt"]["wcr_webb"] = K.wild_score_test_webb(scc, np.random.default_rng(K.SEED + 2_000_000 + k), wild_reps, "webb")
        obs[fold]["cnt"]["wcr_rademacher_check"] = K.wild_score_test_webb(scc, np.random.default_rng(K.SEED + 2_100_000 + k), 999, "rademacher")
        obs[fold]["lpm"]["wcr_webb"] = K.wild_score_test_webb(scl, np.random.default_rng(K.SEED + 2_200_000 + k), wild_reps, "webb")
        obs[fold]["lpm"]["wcr_rademacher_check"] = K.wild_score_test_webb(scl, np.random.default_rng(K.SEED + 2_300_000 + k), 999, "rademacher")
    t0 = time.time()
    res = []
    _W.clear()
    _W.update(payload)  # notebook: sequential loop instead of the spawn process pool
    for j, i in enumerate(range(draws)):
        res.append(_inf_draw(i))
        if (j + 1) % 500 == 0:
            logger.info(f"inference draws {j + 1}/{draws} ({time.time() - t0:.0f}s)")
    dd = pd.DataFrame(res).sort_values("draw").reset_index(drop=True)
    dd.to_csv(out / "inference_draws.csv", index=False)
    rows, summ = [], {"label": LABEL, "draws": draws, "wild_reps": wild_reps, "rows": {}}
    src = f"{K.rel(out)}/inference_rows.csv"

    def put(test, fold, row, value, note=""):
        rows.append({"test": test, "fold": fold, "row": row, "value": value, "note": note, "source": src})

    for fold in samples:
        for mdl, row in (("cnt", "coprimary_A_count"), ("lpm", "K1a_LPM")):
            o = obs[fold][mdl]
            ent = {"t_obs": o["t"], "p_crv1": o["p_crv1"], "G": o["G"]}
            for tag, name in (("plain", "rand_t"), ("fl", "freedman_lane")):
                ts = dd[f"{fold}_{mdl}_{tag}_t"].to_numpy(dtype=float)
                rp = K.rand_p(o["t"], ts)
                ent[f"p_{name}"], ent[f"mc_se_{name}"], ent[f"converged_{name}"] = rp["p"], rp["mc_se"], rp["draws_converged"]
                tsf = ts[np.isfinite(ts)]
                pcr = 2 * stats.t.sf(np.abs(tsf), o["G"] - 1)
                ent[f"crv1_null_rejection_{name}"] = float((pcr < 0.05).mean())
                ent[f"z_sd_null_{name}"] = float(tsf.std())
                ent[f"z_mean_null_{name}"] = float(tsf.mean())
            ent["p_wcr_webb"] = (o["wcr_webb"] or {}).get("p")
            ent["p_wcr_rademacher_check"] = (o["wcr_rademacher_check"] or {}).get("p")
            summ["rows"][f"{fold}.{row}"] = ent
            put("p_crv1", fold, row, o["p_crv1"])
            put("p_rand_t", fold, row, ent["p_rand_t"], "headline p (declared)")
            put("mc_se_rand_t", fold, row, ent["mc_se_rand_t"])
            put("p_freedman_lane", fold, row, ent["p_freedman_lane"])
            put("mc_se_freedman_lane", fold, row, ent["mc_se_freedman_lane"])
            put("p_wcr_webb", fold, row, ent["p_wcr_webb"], f"{wild_reps} Webb draws")
            put("p_wcr_rademacher_check", fold, row, ent["p_wcr_rademacher_check"], "999 Rademacher draws")
            put("crv1_null_rejection", fold, row, ent["crv1_null_rejection_rand_t"], "share of plain-shuffle draws with CRV1 p<0.05")
            put("z_sd_null", fold, row, ent["z_sd_null_rand_t"])
            put("crv1_null_rejection_fl", fold, row, ent["crv1_null_rejection_freedman_lane"])
            put("z_sd_null_fl", fold, row, ent["z_sd_null_freedman_lane"])
            put("draws_converged", fold, row, ent["converged_rand_t"])
    # IVW pooled randomization (same draw index across folds)
    for mdl, row in (("cnt", "coprimary_A_count"), ("lpm", "K1a_LPM")):
        fl = list(samples)
        b = np.array([obs[f][mdl]["b"] * obs[f][mdl]["sd"] for f in fl])
        se = np.array([obs[f][mdl]["se"] * obs[f][mdl]["sd"] for f in fl])
        w = 1 / se ** 2
        z_obs = float((w * b).sum() / math.sqrt(w.sum()))
        ent = {"z_obs": z_obs, "p_normal": float(2 * stats.norm.sf(abs(z_obs)))}
        for tag, name in (("plain", "rand_t"), ("fl", "freedman_lane")):
            B = np.column_stack([dd[f"{f}_{mdl}_{tag}_b"].to_numpy(float) * obs[f][mdl]["sd"] for f in fl])
            S = np.column_stack([dd[f"{f}_{mdl}_{tag}_se"].to_numpy(float) * obs[f][mdl]["sd"] for f in fl])
            ok = np.all(np.isfinite(B) & np.isfinite(S), axis=1)
            W_ = 1 / S[ok] ** 2
            zs = (W_ * B[ok]).sum(1) / np.sqrt(W_.sum(1))
            rp = K.rand_p(z_obs, zs)
            ent[f"p_{name}"], ent[f"mc_se_{name}"], ent[f"converged_{name}"] = rp["p"], rp["mc_se"], rp["draws_converged"]
            ent[f"z_sd_null_{name}"] = float(zs.std())
            ent[f"normal_null_rejection_{name}"] = float((np.abs(zs) > 1.96).mean())
        summ["rows"][f"IVW.{row}"] = ent
        put("p_normal", "IVW", row, ent["p_normal"])
        put("p_rand_t", "IVW", row, ent["p_rand_t"], "same draw index in every fold")
        put("mc_se_rand_t", "IVW", row, ent["mc_se_rand_t"])
        put("p_freedman_lane", "IVW", row, ent["p_freedman_lane"])
        put("z_sd_null", "IVW", row, ent["z_sd_null_rand_t"])
        put("crv1_null_rejection", "IVW", row, ent["normal_null_rejection_rand_t"], "share of draws with |z_IVW*| > 1.96")
        put("p_wcr_webb", "IVW", row, None, "not defined for the pooled z (per-fold wild p reported)")
    K.dump(K.clean(summ), out / "inference_summary.json")
    pd.DataFrame(rows).to_csv(out / "inference_rows.csv", index=False)
    logger.info(f"inference done in {time.time() - t0:.0f}s: " +
                "; ".join(f"{k} p_rand {v.get('p_rand_t'):.4f}" for k, v in summ["rows"].items()))
    return summ

## Run: load the frozen held-out fold and estimate K1

This is the relevant part of `k_run.main()`. The spec hash assertion, the gates file and the K3 pooled sample need files
that are not shipped, so the frozen spec values come from the data file's metadata instead.

In [ ]:
spec = meta["spec"]                     # original: json.loads(K.SPEC_PATH.read_text()) after K.assert_spec_hash()
samples = {"HELDOUT": K.load_fold("HELDOUT")}
out = OUT_DIR
out.mkdir(parents=True, exist_ok=True)
s = samples["HELDOUT"]
print(f"HELDOUT sample: {len(s)} entry events, {s.concept_id.nunique()} concepts; "
      f"share with any uptake P(Y>=1) = {s.Any.mean():.3f}; max Y_strict = {s.Y_strict.max():.0f}")

status = {}
t0 = time.time()
dec = run_k1(samples, spec, out, BOOT_DRAWS)
status["k1"] = {"status": "RUN", "seconds": round(time.time() - t0, 1)}
print(status)

## Run: size-correct inference on the held-out fold

In [ ]:
t0 = time.time()
summ = run_inference(samples, out, RAND_T_DRAWS, WILD_REPS)
status["inference"] = {"status": "RUN", "seconds": round(time.time() - t0, 1)}
print(status)

## Results (1/3): recomputed held-out numbers vs the published artifact

Each point estimate, CI and CRV1 p is deterministic, so it should match the published value exactly.
The bootstrap CIs and randomization / Freedman-Lane / wild-bootstrap p values are Monte Carlo quantities. The demo
uses the original seeds, so at the original draw counts (1,000 / 2,000 / 9,999, the defaults in the config cell) they
reproduce the published values exactly as well. With smaller counts they are noisier approximations.

In [ ]:
ref = meta["reference_results"]
k1 = pd.read_csv(out / "k1_rows.csv")
k1_ref = pd.DataFrame(ref["k1_rows_heldout_and_ivw"])
key = ["test", "outcome", "fe"]
cmp = (k1[k1.fold == "HELDOUT"][key + ["model", "unit", "estimate", "ci_lo", "ci_hi", "p_crv1", "N", "G"]]
       .merge(k1_ref[k1_ref.fold == "HELDOUT"][key + ["estimate", "ci_lo", "ci_hi"]], on=key, how="left",
              suffixes=("", "_published")))
cmp["abs_diff"] = (cmp.estimate - cmp.estimate_published).abs()
pd.set_option("display.width", 200)
print("K1 rows, HELD-OUT fold (demo recomputation vs published)")
print(cmp[["test", "outcome", "unit", "estimate", "ci_lo", "ci_hi", "p_crv1", "N", "G", "estimate_published", "abs_diff"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

d = dec["folds"]["HELDOUT"]
dref = [r for r in ref["k1_rows_heldout_and_ivw"] if r["test"] == "K1c_decomposition"][0]
print(f"\nLog-link decomposition: extensive share = {d['ext_share']:.3f} "
      f"(bootstrap 95% CI {d['ci_ext_share'][0]:.3f}..{d['ci_ext_share'][1]:.3f} from {d['draws_converged']} draws) | "
      f"published {dref['estimate']:.3f} [{dref['ci_lo']:.3f}, {dref['ci_hi']:.3f}] from 1,000 draws")

inf = pd.read_csv(out / "inference_rows.csv")
inf_ref = pd.DataFrame(ref["inference_rows_heldout_and_ivw"])
ci = (inf[inf.fold == "HELDOUT"][["test", "row", "value"]]
      .merge(inf_ref[inf_ref.fold == "HELDOUT"][["test", "row", "value"]], on=["test", "row"], suffixes=("", "_published")))
print(f"\nINFERENCE rows, HELD-OUT fold ({RAND_T_DRAWS} randomization draws, {WILD_REPS} Webb draws in the demo)")
print(ci.pivot(index="test", columns="row", values=["value", "value_published"])
      .to_string(float_format=lambda v: f"{v:.4g}"))

## Results (2/3): the K1 verdict rule applied to the demo

This is the frozen K1 rule from `k/report.py::verdicts`: `BOTH` when the IVW extensive (LPM) CI and the IVW intensive
(K1b) CI both exclude 0; `EXTENSIVE` or `INTENSIVE-ONLY` when only one does; otherwise `UNRESOLVED`. In the demo the "IVW" is
the held-out fold alone, so this is the held-out fold's own verdict. The published verdict pools all three folds.

In [ ]:
iv = dec["ivw"]
e, i = iv["K1a_LPM_pp_per_sd"], iv["K1b_log_irr_per_sd"]
ext = not (e["ci"][0] <= 0 <= e["ci"][1])
itn = not (i["ci"][0] <= 0 <= i["ci"][1])
v = "BOTH" if ext and itn else "EXTENSIVE" if ext else "INTENSIVE-ONLY" if itn else "UNRESOLVED"
p_rand = summ["rows"]["IVW.K1a_LPM"]["p_rand_t"]
print(f"HELD-OUT K1 verdict (demo): {v}")
print(f"  extensive LPM {e['est']:.2f} pp/SD [{e['ci'][0]:.2f}, {e['ci'][1]:.2f}], randomization-t p {p_rand:.4f}")
print(f"  intensive K1b IRR/SD {math.exp(i['est']):.3f} [{math.exp(i['ci'][0]):.3f}, {math.exp(i['ci'][1]):.3f}]")
print("\nPublished verdicts (all three folds / pooled screen+held-out):")
for k, val in ref["verdicts"].items():
    print(f"  {k}: {val}")

## Results (3/3): figures

* **(a)** Threshold ladder on the held-out fold. The effect sits at the *start* of uptake (Y≥1) and fades toward
  `EST_bin`. This explains the earlier held-out EST_bin null.
* **(b)** Randomization null distribution of the held-out count-model t. The observed t is far in the tail. The null SD
  is above 1, the signature of CRV1 over-rejection with few clusters.
* **(c)** K1 extensive and intensive margins per fold and IVW (published, all three folds, with the demo's held-out
  recomputation overlaid).
* **(d)** K3 field boundary (published): the IRR per pooled SD by origin field. The Wald equality p is 0.638, so no boundary is detected.

In [ ]:
OKABE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

# (a) threshold ladder (demo recomputation vs published)
lad = cmp[cmp.test == "K1d_ladder"].set_index("outcome").loc[["Any", "Y_ge3", "Y_ge5", "EST_bin"]]
x = np.arange(len(lad))
ax[0, 0].errorbar(x - 0.08, lad.estimate, yerr=[lad.estimate - lad.ci_lo, lad.ci_hi - lad.estimate], fmt="o",
                  color=OKABE[1], capsize=3, label="demo (held-out)")
ax[0, 0].plot(x + 0.08, lad.estimate_published, "kx", label="published")
ax[0, 0].axhline(0, color="grey", lw=0.8)
ax[0, 0].set_xticks(x, ["Y>=1", "Y>=3", "Y>=5", "EST_bin"])
ax[0, 0].set_ylabel("pp per SD of A_cont (LPM)")
ax[0, 0].set_title("(a) Threshold ladder, held-out fold")
ax[0, 0].legend(frameon=False)

# (b) randomization null of the count-model t
dd = pd.read_csv(out / "inference_draws.csv")
t_null = dd["HELDOUT_cnt_plain_t"].dropna()
t_obs = summ["rows"]["HELDOUT.coprimary_A_count"]["t_obs"]
ax[0, 1].hist(t_null, bins=max(5, min(40, len(t_null) // 5)), color=OKABE[4], edgecolor="white")
ax[0, 1].axvline(t_obs, color=OKABE[5], lw=2, label=f"observed t = {t_obs:.2f}")
ax[0, 1].axvline(-t_obs, color=OKABE[5], lw=1, ls="--")
r = summ["rows"]["HELDOUT.coprimary_A_count"]
ax[0, 1].set_title(f"(b) Randomization-t null, held-out count row\n"
                   f"p_rand = {r['p_rand_t']:.4f} ({len(t_null)} draws), null SD = {r['z_sd_null_rand_t']:.2f}")
ax[0, 1].set_xlabel("t* under within-concept permutation of A_cont")
ax[0, 1].legend(frameon=False)

# (c) K1 margins per fold (published) + demo held-out
allk = pd.DataFrame(ref["k1_rows_all_folds_main"])
ivw_rows = pd.DataFrame(ref["k1_rows_heldout_and_ivw"])
folds = ["SCREEN", "HELDOUT", "MESH", "IVW"]
ext_pub = [allk[(allk.test == "K1a_LPM") & (allk.fold == f)].iloc[0] if f != "IVW" else
           ivw_rows[ivw_rows.test == "K1f_IVW_K1a_LPM_pp_per_sd"].iloc[0] for f in folds]
int_pub = [allk[(allk.test == "K1b") & (allk.fold == f)].iloc[0] if f != "IVW" else
           ivw_rows[ivw_rows.test == "K1f_IVW_K1b_log_irr_per_sd"].iloc[0] for f in folds]
y = np.arange(len(folds))[::-1]
a2 = ax[1, 0].twiny()
for yy, r_e, r_i, f in zip(y, ext_pub, int_pub, folds):
    ax[1, 0].errorbar(r_e.estimate, yy + 0.12, xerr=[[r_e.estimate - r_e.ci_lo], [r_e.ci_hi - r_e.estimate]], fmt="o",
                      color=OKABE[0], capsize=3)
    a2.errorbar(r_i.estimate, yy - 0.12, xerr=[[r_i.estimate - r_i.ci_lo], [r_i.ci_hi - r_i.estimate]], fmt="s",
                color=OKABE[2], capsize=3)
demo_ext = cmp[cmp.test == "K1a_LPM"].iloc[0]
demo_int = cmp[cmp.test == "K1b"].iloc[0]
ax[1, 0].plot(demo_ext.estimate, y[1] + 0.12, "*", ms=13, color=OKABE[5], label="demo recomputation")
a2.plot(demo_int.estimate, y[1] - 0.12, "*", ms=13, color=OKABE[5])
ax[1, 0].set_yticks(y, folds)
ax[1, 0].axvline(0, color=OKABE[0], lw=0.6, ls=":")
a2.axvline(1, color=OKABE[2], lw=0.6, ls=":")
ax[1, 0].set_xlabel("extensive: pp per SD (circles, bottom axis)", color=OKABE[0])
a2.set_xlabel("intensive: IRR per SD on Y>=1 (squares, top axis)", color=OKABE[2])
ax[1, 0].set_title("(c) K1 margins by fold (published) + demo", pad=28)
ax[1, 0].legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, 0.93))

# (d) K3 field boundary (published)
k3 = pd.DataFrame(ref["k3_rows"])
g = k3[k3.test.isin(["K3_group", "K3_MESH_reference"])].reset_index(drop=True)
yy = np.arange(len(g))[::-1]
ax[1, 1].errorbar(g.estimate, yy, xerr=[g.estimate - g.ci_lo, g.ci_hi - g.estimate], fmt="o", color=OKABE[3], capsize=3)
ax[1, 1].axvline(1, color="grey", lw=0.8)
short = {"MeSH biomedicine->biomedicine (R2)": "MeSH reference (R2)"}
ax[1, 1].set_yticks(yy, [f"{short.get(r_, r_)}\n(G={int(G_)}, N={int(N_)})" for r_, G_, N_ in zip(g.row, g.G, g.N)])
w = k3[k3.test == "K3_wald_equality"].iloc[0]
c = k3[k3.test == "K3_phys_contrast"].iloc[0]
ax[1, 1].set_title(f"(d) K3 field boundary (published)\nWald p {w.p_crv1:.3f}, perm p {w.p_perm:.3f}; "
                   f"phys/rest ratio {c.estimate:.3f} [{c.ci_lo:.2f}, {c.ci_hi:.2f}]")
ax[1, 1].set_xlabel("IRR per pooled SD of A_cont")
plt.tight_layout()
plt.show()